# BirdCLEF+ 2026 — Inference Notebook
Auto-generated by structured search agent.
Config: depth=5 | filt=48-constant | lr=0.0030336987857336314 | adam | drop=0.09 | BN=Y | res=N | mels=128 | frames=192 | bs=8


In [ ]:
import os, numpy as np, pandas as pd, librosa, tensorflow as tf
from pathlib import Path

SR = 32000
CLIP_SECONDS = 5.0
N_MELS = 128
N_FRAMES = 192
N_FFT = 1024
HOP_LENGTH = 512

COMP_DIR = '/kaggle/input/birdclef-2026'
MODEL_DIR = '/kaggle/input/birdclef-model'


In [ ]:
model = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'model.keras'))
sample_sub = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))
species_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'Model loaded. Species: {len(species_cols)}')


In [ ]:
test_dir = Path(COMP_DIR) / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg')) if test_dir.exists() else []
print(f'Test soundscapes: {len(test_files)}')

all_rows = []
seg_samples = int(SR * CLIP_SECONDS)

for fpath in test_files:
    name = fpath.stem
    y_full, _ = librosa.load(str(fpath), sr=SR, mono=True)
    for start in range(0, len(y_full), seg_samples):
        seg = y_full[start:start+seg_samples]
        if len(seg) < seg_samples:
            seg = np.pad(seg, (0, seg_samples - len(seg)))
        end_sec = (start // seg_samples + 1) * CLIP_SECONDS
        mel = librosa.feature.melspectrogram(
            y=seg, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH, power=2.0)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_r = tf.image.resize(mel_db[..., np.newaxis], (N_MELS, N_FRAMES)).numpy().astype(np.float32)
        preds = model.predict(mel_r[np.newaxis, ...], verbose=0)[0]
        row = {'row_id': f'{name}_{end_sec}'}
        for col, p in zip(species_cols, preds):
            row[col] = float(p)
        all_rows.append(row)

print(f'Prediction rows: {len(all_rows)}')


In [ ]:
sub = pd.DataFrame(all_rows) if all_rows else sample_sub.copy()
sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')
sub[species_cols] = sub[species_cols].fillna(0.0).clip(0.0, 1.0)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Submission saved: {sub.shape}')
